In [ ]:
# !pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.2/471.2 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.79
    Uninstalling langchain-core-0.3.79:
      Successfully uninstalled langchain-core-0.3.79
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.11
    Uninstalling langchain-text-splitters-0.3.11:
      Successfully uninstalled langchain-text-splitters-0.3.11
ERROR: pip's dependency resolver

In [ ]:
!pip install requests beautifulsoup4 faiss-cpu sentence-transformers transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 18.6 MB/s eta 0:00:00


In [ ]:
import requests
from bs4 import BeautifulSoup
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline


In [ ]:
def load_webpage(url):
    """Fetch the webpage and return only readable text."""
    print(f"Loading from: {url}")
    response = requests.get(url)
    response.raise_for_status()  # stop if there’s an error

    soup = BeautifulSoup(response.text, "html.parser")

    # Remove scripts, styles, navbars, etc.
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()

    # Try to get the main article if possible
    article = soup.find("article")
    if article:
        text = article.get_text(separator=" ")
    else:
        text = soup.get_text(separator=" ")

    # Clean up spaces
    return " ".join(text.split())


In [ ]:
def make_chunks(text, chunk_size=500, overlap=50):
    """Split long text into smaller overlapping pieces."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
        if i + chunk_size >= len(words):
            break
    return chunks

In [ ]:
class SimpleVectorStore:
    def __init__(self):
        self.model = SentenceTransformer("all-MiniLM-L6-v2")
        self.index = None
        self.text_chunks = []

    def add_texts(self, chunks):
        """Turn text chunks into vectors and add them to FAISS index."""
        embeddings = self.model.encode(chunks).astype("float32")

        # Create FAISS index if not exists
        if self.index is None:
            self.index = faiss.IndexFlatL2(embeddings.shape[1])

        self.index.add(embeddings)
        self.text_chunks.extend(chunks)

    def search(self, query, k=3):
        """Find top-k similar chunks for a query."""
        query_vec = self.model.encode([query]).astype("float32")
        distances, ids = self.index.search(query_vec, k)
        return [self.text_chunks[i] for i in ids[0]]


In [ ]:
generator = pipeline("text2text-generation", model="google/flan-t5-small")

def answer_question(query, related_chunks):
    """Use the retrieved text to answer the question."""
    context = "\n\n".join(related_chunks)
    prompt = f"Use the context below to answer the question.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    result = generator(prompt, max_new_tokens=200, do_sample=False)
    return result[0]["generated_text"]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [ ]:
import textwrap

def run_rag(url, question):
    # Step 1: Load and clean webpage
    text = load_webpage(url)

    # Step 2: Split into chunks
    chunks = make_chunks(text)

    # Step 3: Create vector store
    store = SimpleVectorStore()
    store.add_texts(chunks)

    # Step 4: Search for relevant chunks
    related = store.search(question)

    # Step 5: Print retrieved chunks clearly (wrapped, no horizontal scroll)
    print("\n--- Retrieved Chunks ---\n")
    for i, chunk in enumerate(related, 1):
        wrapped = textwrap.fill(chunk, width=100)  # wrap at 100 chars
        print(f"Chunk {i}:\n{wrapped}\n")

    # Step 6: Generate the final answer
    answer = answer_question(question, related)
    print("\n💬 Final Answer:\n")
    print(textwrap.fill(answer, width=100))


In [ ]:
url = "https://github.com/resources/articles/what-is-agentic-ai"
question = "How does agentic AI work??"
run_rag(url, question)

Loading from: https://github.com/resources/articles/what-is-agentic-ai


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1711 > 512). Running this sequence through the model will result in indexing errors



--- Retrieved Chunks ---

Chunk 1:
What is Agentic AI? · GitHub Skip to content Explore GitHub's latest Universe launches — join the
Product Roadmap webinar You signed in with another tab or window. Reload to refresh your session.
You signed out in another tab or window. Reload to refresh your session. You switched accounts on
another tab or window. Reload to refresh your session. Dismiss alert What is Agentic AI? At its
core, agentic AI is a system built around an AI model that enables it to operate more like an active
teammate, capable of setting goals, taking action, and working independently. This system uses
memory (to retain context), tools (like access to your codebase or terminal), a defined goal, and
the autonomy to act. The model provides the reasoning; the surrounding system transforms that
reasoning into purposeful execution, allowing the agent to plan, adapt, and complete tasks with
minimal human input. Today, teams are starting to explore agentic AI in scenarios such as 